# Reverse Strategy: One-vs-All Theme Classifiers
## 12 Independent Binary Classifiers → Aggregate for Final Prediction

### Strategy:
1. Train 12 binary classifiers (one per theme)
2. Each classifier: "Is this Theme X (Accepted)?" vs "All others"
3. At test time: Run all 12, aggregate predictions
4. Decision: Highest confidence theme → Accept with that theme
5. All low confidence → Reject

### Advantages:
- ✅ No error propagation from Stage 1 to Stage 2
- ✅ Each classifier learns ONE theme deeply
- ✅ Implicit rejection (no separate reject classifier needed)
- ✅ Per-theme threshold tuning
- ✅ Better for rare themes (3 vs 1716 is easier than 3 vs 195)

### Expected Results:
- **Accept Rate**: 10-20% (similar to training)
- **Theme Distribution**: Mirrors training distribution
- **F1 per Theme**: Varies (high for common, lower for rare)

In [1]:
!pip install -q transformers==4.45.0 scikit-learn openpyxl pandas numpy torch

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 51.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 36.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 94.4 MB/s eta 0:00:00


In [2]:
import os
import gc
import warnings
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import random

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer, AutoModel, AutoConfig,
    get_cosine_schedule_with_warmup
)

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    f1_score, accuracy_score, classification_report,
    precision_recall_curve, roc_auc_score
)

warnings.filterwarnings('ignore')

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed(42)

print('✓ Libraries loaded')
print(f'PyTorch: {torch.__version__}')
print(f'Device: {"GPU - " + torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

✓ Libraries loaded
PyTorch: 2.9.0+cu126
Device: GPU - Tesla P100-PCIE-16GB


In [3]:
class CFG:
    # Paths
    train_path = '/kaggle/input/climate-text-dataset/Human labelled_DTU.xlsx'
    test_path = '/kaggle/input/climate-text-dataset/Master file_10k papers.xlsx'
    output_dir = '/kaggle/working/'
    
    # Model
    model_name = 'microsoft/deberta-v3-base'
    max_length = 384  # Reduced for memory
    
    # Training (for each theme classifier)
    n_folds = 3  # Reduced folds since training 12 models
    n_epochs = 15
    batch_size = 4
    lr = 1e-5
    dropout = 0.25
    
    # Oversampling strategy
    min_positive_samples = 30  # Oversample minority to at least 30
    undersample_negative_ratio = 3  # Keep negative at most 3x positive
    
    # Decision thresholds
    accept_threshold = 0.3  # Minimum confidence to consider a theme
    multi_label_threshold = 0.5  # If multiple themes > this, pick highest
    
    # General
    weight_decay = 0.08
    warmup_ratio = 0.1
    max_grad_norm = 1.0
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    fp16 = True
    num_workers = 0
    early_stopping_patience = 4
    seed = 42

print('✓ Configuration set for ONE-VS-ALL strategy')

✓ Configuration set for ONE-VS-ALL strategy


In [4]:
# Load data
print("Loading data...")

train_df = pd.read_excel(CFG.train_path, skiprows=1)
train_df.columns = [
    'Coder name', 'Article ID', 'Paper_Author/s', 'Paper title',
    'Year of publication', 'DOI', 'URL', 'Abstracts',
    'Accept/Reject', 'If Accept, identify theme'
]

train_df = train_df[train_df['Accept/Reject'].isin(['Accept', 'Reject'])].copy()
train_df['text'] = train_df['Abstracts'].fillna('')
train_df = train_df[train_df['text'].str.len() > 50].reset_index(drop=True)

# Get themes
accepted_df = train_df[train_df['Accept/Reject'] == 'Accept'].copy()
accepted_df = accepted_df[accepted_df['If Accept, identify theme'].notna()].copy()

theme_encoder = LabelEncoder()
accepted_df['theme_id'] = theme_encoder.fit_transform(accepted_df['If Accept, identify theme'])

# Create binary labels for each theme (one-vs-all)
num_themes = len(theme_encoder.classes_)

for theme_id in range(num_themes):
    # Label: 1 if this theme, 0 otherwise
    train_df[f'is_theme_{theme_id}'] = 0
    
    # Set 1 for papers with this theme
    theme_papers = accepted_df[accepted_df['theme_id'] == theme_id].index
    train_df.loc[theme_papers, f'is_theme_{theme_id}'] = 1

# Test data
test_df = pd.read_excel(CFG.test_path)
test_df['text'] = test_df['Abstract'].fillna('')
test_df = test_df[test_df['text'].str.len() > 50].reset_index(drop=True)

print(f"\n{'='*80}")
print("DATA LOADED - ONE-VS-ALL STRATEGY")
print(f"{'='*80}")
print(f"\nTraining papers: {len(train_df)}")
print(f"Test papers: {len(test_df)}")
print(f"\nThemes (12):")
for theme_id, theme_name in enumerate(theme_encoder.classes_):
    n_positive = (train_df[f'is_theme_{theme_id}'] == 1).sum()
    n_negative = (train_df[f'is_theme_{theme_id}'] == 0).sum()
    print(f"  {theme_id:2d}: {n_positive:3d} positive, {n_negative:4d} negative (1:{n_negative/n_positive:.1f}) - {theme_name}")
print(f"{'='*80}")

Loading data...

DATA LOADED - ONE-VS-ALL STRATEGY

Training papers: 1719
Test papers: 10175

Themes (12):
   0:   6 positive, 1713 negative (1:285.5) - Access to Services and Wellbeing
   1:   7 positive, 1712 negative (1:244.6) - Better Well-being Metric
   2:  84 positive, 1635 negative (1:19.5) - Climate change mitigation options & Well-being
   3:   6 positive, 1713 negative (1:285.5) - Ecosystem & Non-human Well-being
   4:  31 positive, 1688 negative (1:54.5) - Empowering People & Governance, Policy
   5:   8 positive, 1711 negative (1:213.9) - Equity, Social Justice, Just Transition
   6:  21 positive, 1698 negative (1:80.9) - Implemented mitigation Options and Well-being
   7:   6 positive, 1713 negative (1:285.5) - Integrated Responses and Well-being
   8:  10 positive, 1709 negative (1:170.9) - SDGs and Well-being
   9:   3 positive, 1716 negative (1:572.0) - Systems Thinking- Modelling and Supply Chain
  10:   5 positive, 1714 negative (1:342.8) - Well-being Loss from Clima

In [5]:
class ClimateDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        encoding = self.tokenizer(
            str(self.texts[idx]),
            add_special_tokens=True,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'label': torch.tensor(self.labels[idx], dtype=torch.long)
        }

class ThemeBinaryClassifier(nn.Module):
    """Binary classifier for one-vs-all theme detection"""
    def __init__(self, model_name, dropout=0.25):
        super().__init__()
        self.config = AutoConfig.from_pretrained(model_name)
        self.config.update({
            'hidden_dropout_prob': dropout,
            'attention_probs_dropout_prob': dropout,
        })
        
        self.transformer = AutoModel.from_pretrained(model_name, config=self.config)
        hidden_size = self.config.hidden_size
        
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 2)  # Binary: theme or not
        )
    
    def forward(self, input_ids, attention_mask):
        outputs = self.transformer(input_ids=input_ids, attention_mask=attention_mask)
        pooled = outputs.last_hidden_state[:, 0]
        return self.classifier(pooled)

print("✓ Model architecture defined")

✓ Model architecture defined


In [6]:
def train_one_theme_classifier(theme_id, theme_name, train_df, tokenizer, CFG):
    """
    Train a single one-vs-all binary classifier for one theme
    """
    print(f"\n{'='*80}")
    print(f"TRAINING CLASSIFIER FOR THEME {theme_id}: {theme_name}")
    print(f"{'='*80}")
    
    # Get positive and negative samples
    positive_samples = train_df[train_df[f'is_theme_{theme_id}'] == 1].copy()
    negative_samples = train_df[train_df[f'is_theme_{theme_id}'] == 0].copy()
    
    n_positive_orig = len(positive_samples)
    n_negative_orig = len(negative_samples)
    
    print(f"Original: {n_positive_orig} positive, {n_negative_orig} negative (1:{n_negative_orig/max(n_positive_orig,1):.1f})")
    
    # Oversample positive to minimum threshold
    if n_positive_orig < CFG.min_positive_samples:
        n_needed = CFG.min_positive_samples - n_positive_orig
        positive_extra = positive_samples.sample(n=n_needed, replace=True, random_state=CFG.seed)
        positive_samples = pd.concat([positive_samples, positive_extra], ignore_index=True)
    
    # Undersample negative to ratio
    target_negative = min(len(positive_samples) * CFG.undersample_negative_ratio, n_negative_orig)
    negative_samples = negative_samples.sample(n=int(target_negative), random_state=CFG.seed)
    
    # Combine and shuffle
    theme_train_df = pd.concat([positive_samples, negative_samples], ignore_index=True)
    theme_train_df = theme_train_df.sample(frac=1, random_state=CFG.seed).reset_index(drop=True)
    
    print(f"Balanced: {len(positive_samples)} positive, {len(negative_samples)} negative (1:{len(negative_samples)/len(positive_samples):.1f})")
    
    # K-Fold training
    skf = StratifiedKFold(n_splits=CFG.n_folds, shuffle=True, random_state=CFG.seed)
    
    fold_models = []
    fold_thresholds = []
    oof_probs = np.zeros(len(theme_train_df))
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(theme_train_df, theme_train_df[f'is_theme_{theme_id}'])):
        print(f"\n  Fold {fold+1}/{CFG.n_folds}")
        
        fold_train = theme_train_df.iloc[train_idx]
        fold_val = theme_train_df.iloc[val_idx]
        
        # Datasets
        train_dataset = ClimateDataset(
            fold_train['text'].values,
            fold_train[f'is_theme_{theme_id}'].values,
            tokenizer,
            CFG.max_length
        )
        
        val_dataset = ClimateDataset(
            fold_val['text'].values,
            fold_val[f'is_theme_{theme_id}'].values,
            tokenizer,
            CFG.max_length
        )
        
        train_loader = DataLoader(train_dataset, batch_size=CFG.batch_size, shuffle=True, num_workers=CFG.num_workers)
        val_loader = DataLoader(val_dataset, batch_size=CFG.batch_size*2, shuffle=False, num_workers=CFG.num_workers)
        
        # Model
        model = ThemeBinaryClassifier(CFG.model_name, dropout=CFG.dropout).to(CFG.device)
        
        # Weighted loss
        pos_weight = len(negative_samples) / max(len(positive_samples), 1)
        class_weights = torch.tensor([1.0, min(pos_weight, 10.0)], dtype=torch.float).to(CFG.device)  # Cap at 10
        criterion = nn.CrossEntropyLoss(weight=class_weights)
        
        # Optimizer
        optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.lr, weight_decay=CFG.weight_decay)
        num_steps = len(train_loader) * CFG.n_epochs
        scheduler = get_cosine_schedule_with_warmup(
            optimizer,
            num_warmup_steps=int(num_steps * CFG.warmup_ratio),
            num_training_steps=num_steps
        )
        
        scaler = torch.cuda.amp.GradScaler() if CFG.fp16 else None
        
        # Training
        best_f1 = 0
        patience = 0
        best_state = model.state_dict().copy()  # ← CRITICAL FIX: Initialize
        best_val_probs = None
        
        for epoch in range(CFG.n_epochs):
            # Train
            model.train()
            for batch in train_loader:
                input_ids = batch['input_ids'].to(CFG.device)
                attention_mask = batch['attention_mask'].to(CFG.device)
                labels = batch['label'].to(CFG.device)
                
                if scaler:
                    with torch.cuda.amp.autocast():
                        logits = model(input_ids, attention_mask)
                        loss = criterion(logits, labels)
                    scaler.scale(loss).backward()
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.max_grad_norm)
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    logits = model(input_ids, attention_mask)
                    loss = criterion(logits, labels)
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.max_grad_norm)
                    optimizer.step()
                
                optimizer.zero_grad()
                scheduler.step()
            
            # Validate
            model.eval()
            val_preds, val_probs_list, val_labels = [], [], []
            
            with torch.no_grad():
                for batch in val_loader:
                    input_ids = batch['input_ids'].to(CFG.device)
                    attention_mask = batch['attention_mask'].to(CFG.device)
                    labels = batch['label'].to(CFG.device)
                    
                    logits = model(input_ids, attention_mask)
                    probs = F.softmax(logits, dim=1).cpu().numpy()
                    preds = np.argmax(probs, axis=1)
                    
                    val_probs_list.append(probs)
                    val_preds.extend(preds)
                    val_labels.extend(labels.cpu().numpy())
            
            val_probs_concat = np.vstack(val_probs_list)
            val_f1 = f1_score(val_labels, val_preds, average='binary', zero_division=0)
            
            if epoch % 3 == 0:
                print(f"    Epoch {epoch+1:2d}: F1={val_f1:.4f}")
            
            if val_f1 > best_f1:
                best_f1 = val_f1
                best_state = model.state_dict().copy()
                best_val_probs = val_probs_concat.copy()
                patience = 0
            else:
                patience += 1
                if patience >= CFG.early_stopping_patience:
                    print(f"    Early stopping at epoch {epoch+1}")
                    break
        
        # Load best
        model.load_state_dict(best_state)
        
        # Find optimal threshold
        if best_val_probs is not None:
            precisions, recalls, thresholds = precision_recall_curve(val_labels, best_val_probs[:, 1])
            f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-8)
            best_idx = np.argmax(f1_scores)
            best_threshold = thresholds[best_idx] if best_idx < len(thresholds) else 0.5
            oof_probs[val_idx] = best_val_probs[:, 1]
        else:
            best_threshold = 0.5
            oof_probs[val_idx] = val_probs_concat[:, 1]
        
        # Move to CPU and save
        model.cpu()
        fold_models.append(model)
        fold_thresholds.append(best_threshold)
        
        print(f"    Best F1: {best_f1:.4f}, Threshold: {best_threshold:.4f}")
        
        # Cleanup
        del train_dataset, val_dataset, train_loader, val_loader
        gc.collect()
        torch.cuda.empty_cache()
    
    # Overall OOF evaluation
    avg_threshold = np.mean(fold_thresholds)
    oof_preds = (oof_probs >= avg_threshold).astype(int)
    oof_f1 = f1_score(theme_train_df[f'is_theme_{theme_id}'].values, oof_preds, average='binary', zero_division=0)
    
    print(f"\n✓ Theme {theme_id} Complete: OOF F1={oof_f1:.4f}, Avg Threshold={avg_threshold:.4f}")
    
    return {
        'theme_id': theme_id,
        'theme_name': theme_name,
        'models': fold_models,
        'thresholds': fold_thresholds,
        'avg_threshold': avg_threshold,
        'oof_f1': oof_f1
    }

## Train All 12 Theme Classifiers

In [7]:
print("\n" + "="*80)
print("TRAINING 12 ONE-VS-ALL THEME CLASSIFIERS")
print("="*80)

tokenizer = AutoTokenizer.from_pretrained(CFG.model_name)

all_theme_classifiers = []

for theme_id in range(num_themes):
    theme_name = theme_encoder.classes_[theme_id]
    
    classifier_info = train_one_theme_classifier(
        theme_id=theme_id,
        theme_name=theme_name,
        train_df=train_df,
        tokenizer=tokenizer,
        CFG=CFG
    )
    
    all_theme_classifiers.append(classifier_info)
    
    # Save memory
    gc.collect()
    torch.cuda.empty_cache()

print("\n" + "="*80)
print("✓ ALL 12 CLASSIFIERS TRAINED")
print("="*80)

print("\nOOF F1 Scores per Theme:")
for clf in all_theme_classifiers:
    print(f"  Theme {clf['theme_id']:2d}: F1={clf['oof_f1']:.4f}, Threshold={clf['avg_threshold']:.4f} - {clf['theme_name']}")


TRAINING 12 ONE-VS-ALL THEME CLASSIFIERS


tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]


TRAINING CLASSIFIER FOR THEME 0: Access to Services and Wellbeing
Original: 6 positive, 1713 negative (1:285.5)
Balanced: 30 positive, 90 negative (1:3.0)

  Fold 1/3


pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

    Epoch  1: F1=0.4000
    Epoch  4: F1=0.8235
    Epoch  7: F1=0.8889
    Early stopping at epoch 9
    Best F1: 0.8889, Threshold: 0.9805

  Fold 2/3
    Epoch  1: F1=0.4000
    Epoch  4: F1=0.6667
    Epoch  7: F1=0.9524
    Epoch 10: F1=1.0000
    Epoch 13: F1=1.0000
    Early stopping at epoch 13
    Best F1: 1.0000, Threshold: 0.9884

  Fold 3/3
    Epoch  1: F1=0.4000
    Epoch  4: F1=0.9524
    Epoch  7: F1=0.9524
    Early stopping at epoch 8
    Best F1: 0.9524, Threshold: 0.5921

✓ Theme 0 Complete: OOF F1=0.7500, Avg Threshold=0.8536

TRAINING CLASSIFIER FOR THEME 1: Better Well-being Metric
Original: 7 positive, 1712 negative (1:244.6)
Balanced: 30 positive, 90 negative (1:3.0)

  Fold 1/3
    Epoch  1: F1=0.0000
    Epoch  4: F1=0.0000
    Early stopping at epoch 4
    Best F1: 0.0000, Threshold: 0.5000

  Fold 2/3
    Epoch  1: F1=0.4000
    Epoch  4: F1=0.5714
    Epoch  7: F1=0.5405
    Epoch 10: F1=0.4545
    Early stopping at epoch 10
    Best F1: 0.6452, Threshold:

## Test Predictions: Run All 12 Classifiers and Aggregate

In [8]:
print("\n" + "="*80)
print("TEST PREDICTIONS - AGGREGATING 12 CLASSIFIERS")
print("="*80)

# Create test dataset
test_dataset = ClimateDataset(
    test_df['text'].values,
    np.zeros(len(test_df)),
    tokenizer,
    CFG.max_length
)

test_loader = DataLoader(test_dataset, batch_size=CFG.batch_size*2, shuffle=False, num_workers=CFG.num_workers)

# Get predictions from all 12 classifiers
all_theme_probs = np.zeros((len(test_df), num_themes))

for theme_id, clf_info in enumerate(all_theme_classifiers):
    print(f"\nPredicting with Theme {theme_id} classifier...")
    
    # Ensemble predictions from all folds
    theme_probs_all_folds = []
    
    for model in clf_info['models']:
        model.to(CFG.device)
        model.eval()
        
        theme_probs = []
        
        with torch.no_grad():
            for batch in test_loader:
                input_ids = batch['input_ids'].to(CFG.device)
                attention_mask = batch['attention_mask'].to(CFG.device)
                
                logits = model(input_ids, attention_mask)
                probs = F.softmax(logits, dim=1).cpu().numpy()
                theme_probs.append(probs[:, 1])  # Probability of being THIS theme
        
        theme_probs_all_folds.append(np.concatenate(theme_probs))
        
        model.cpu()
        torch.cuda.empty_cache()
    
    # Average across folds
    all_theme_probs[:, theme_id] = np.mean(theme_probs_all_folds, axis=0)
    
    print(f"  Mean probability: {all_theme_probs[:, theme_id].mean():.4f}")
    print(f"  Papers > threshold ({clf_info['avg_threshold']:.2f}): {(all_theme_probs[:, theme_id] > clf_info['avg_threshold']).sum()}")

print("\n✓ All 12 classifiers applied")


TEST PREDICTIONS - AGGREGATING 12 CLASSIFIERS

Predicting with Theme 0 classifier...
  Mean probability: 0.0651
  Papers > threshold (0.85): 60

Predicting with Theme 1 classifier...
  Mean probability: 0.4530
  Papers > threshold (0.54): 1402

Predicting with Theme 2 classifier...
  Mean probability: 0.3233
  Papers > threshold (0.68): 670

Predicting with Theme 3 classifier...
  Mean probability: 0.0061
  Papers > threshold (0.59): 25

Predicting with Theme 4 classifier...
  Mean probability: 0.4058
  Papers > threshold (0.52): 3708

Predicting with Theme 5 classifier...
  Mean probability: 0.4022
  Papers > threshold (0.52): 3860

Predicting with Theme 6 classifier...
  Mean probability: 0.4582
  Papers > threshold (0.50): 186

Predicting with Theme 7 classifier...
  Mean probability: 0.2157
  Papers > threshold (0.67): 86

Predicting with Theme 8 classifier...
  Mean probability: 0.4449
  Papers > threshold (0.55): 2782

Predicting with Theme 9 classifier...
  Mean probability: 0.

In [9]:
print("\n" + "="*80)
print("AGGREGATING PREDICTIONS - DECISION LOGIC")
print("="*80)

# Decision logic
max_probs = np.max(all_theme_probs, axis=1)
max_theme_ids = np.argmax(all_theme_probs, axis=1)

# Assign predictions
predictions = []
predicted_themes = []
confidences = []

for i in range(len(test_df)):
    max_prob = max_probs[i]
    max_theme_id = max_theme_ids[i]
    
    # Get the threshold for this theme
    theme_threshold = all_theme_classifiers[max_theme_id]['avg_threshold']
    
    # Decision: If max probability > theme-specific threshold AND > global accept threshold
    if max_prob > theme_threshold and max_prob > CFG.accept_threshold:
        predictions.append('Accept')
        predicted_themes.append(theme_encoder.classes_[max_theme_id])
        confidences.append(max_prob)
    else:
        predictions.append('Reject')
        predicted_themes.append('')
        confidences.append(1 - max_prob)  # Confidence in rejection

# Add to dataframe
test_df['Prediction'] = predictions
test_df['Theme'] = predicted_themes
test_df['Confidence'] = confidences

# Statistics
n_accept = (test_df['Prediction'] == 'Accept').sum()
n_reject = (test_df['Prediction'] == 'Reject').sum()

print(f"\n{'='*80}")
print("FINAL PREDICTIONS")
print(f"{'='*80}")
print(f"\nTotal papers: {len(test_df):,}")
print(f"Predicted Accept: {n_accept:,} ({n_accept/len(test_df)*100:.2f}%)")
print(f"Predicted Reject: {n_reject:,} ({n_reject/len(test_df)*100:.2f}%)")

if n_accept > 0:
    print(f"\nTheme Distribution (Accepted papers):")
    theme_dist = test_df[test_df['Prediction']=='Accept']['Theme'].value_counts()
    for theme, count in theme_dist.items():
        print(f"  {count:4d} ({count/n_accept*100:5.1f}%) - {theme}")
    
    print(f"\nNumber of different themes predicted: {len(theme_dist)}/12")
    
    if len(theme_dist) < 6:
        print(f"\n⚠️ Warning: Only {len(theme_dist)} themes predicted. Consider lowering accept_threshold.")
    else:
        print(f"\n✓ Good diversity: {len(theme_dist)} different themes predicted!")

print(f"\n{'='*80}")


AGGREGATING PREDICTIONS - DECISION LOGIC

FINAL PREDICTIONS

Total papers: 10,175
Predicted Accept: 6,986 (68.66%)
Predicted Reject: 3,189 (31.34%)

Theme Distribution (Accepted papers):
  2555 ( 36.6%) - Equity, Social Justice, Just Transition
  2248 ( 32.2%) - Well-being Loss from Climate Mitigation
  1038 ( 14.9%) - SDGs and Well-being
   357 (  5.1%) - Climate change mitigation options & Well-being
   315 (  4.5%) - Well-being, Community & Societal Systems Thinking
   178 (  2.5%) - Empowering People & Governance, Policy
   101 (  1.4%) - Better Well-being Metric
    63 (  0.9%) - Systems Thinking- Modelling and Supply Chain
    53 (  0.8%) - Access to Services and Wellbeing
    47 (  0.7%) - Integrated Responses and Well-being
    19 (  0.3%) - Ecosystem & Non-human Well-being
    12 (  0.2%) - Implemented mitigation Options and Well-being

Number of different themes predicted: 12/12

✓ Good diversity: 12 different themes predicted!



In [10]:
# Save predictions
output_cols = ['ID_New', 'Article Title', 'Prediction', 'Confidence', 'Theme']
test_df[output_cols].to_csv(f'{CFG.output_dir}/reverse_strategy_predictions.csv', index=False)

print("✓ Predictions saved to: reverse_strategy_predictions.csv")
print(f"\nColumns:")
print(f"  - ID_New: Paper ID")
print(f"  - Article Title: Paper title")
print(f"  - Prediction: Accept or Reject")
print(f"  - Confidence: Confidence score (0-1)")
print(f"  - Theme: Theme name (only for Accept)")

# Display sample
print(f"\nSample predictions (Accepted):")
display(test_df[test_df['Prediction']=='Accept'][output_cols])

print(f"\n{'='*80}")
print("✓ REVERSE STRATEGY (ONE-VS-ALL) COMPLETE!")
print(f"{'='*80}")

✓ Predictions saved to: reverse_strategy_predictions.csv

Columns:
  - ID_New: Paper ID
  - Article Title: Paper title
  - Prediction: Accept or Reject
  - Confidence: Confidence score (0-1)
  - Theme: Theme name (only for Accept)

Sample predictions (Accepted):


,ID_New,Article Title,Prediction,Confidence,Theme
0,OA_3712,NaN,Accept,0.857219,Systems Thinking- Modelling and Supply Chain
1,WoS_1385,"It ' s one thing after another, after another...",Accept,0.917544,"Equity, Social Justice, Just Transition"
2,Scopus_5109,"""A Return to and of the Land"": Indigenous Know...",Accept,0.812806,"Equity, Social Justice, Just Transition"
3,Scopus_4859,"""I see my culture starting to disappear"": Anis...",Accept,0.989308,"Equity, Social Justice, Just Transition"
4,Scopus_1176,"""Impact of Climate Change on Coastal Cities: A...",Accept,0.971161,"Equity, Social Justice, Just Transition"
...,...,...,...,...,...
10165,WoS_2311,Youths' Investigations of Critical Urban Fores...,Accept,0.658588,"Equity, Social Justice, Just Transition"
10166,Scopus_0308,Youths’ Investigations of Critical Urban Fores...,Accept,0.658405,"Equity, Social Justice, Just Transition"
10169,Scopus_3972,Zoos and aquaria: dark tourism or light fun? A...,Accept,0.991705,"Equity, Social Justice, Just Transition"
10170,OA_4235,ҚАЗАҚСТАН РЕСПУБЛИКАСЫНДАҒЫ АЗЫҚ-ТҮЛІК ЖҮЙЕСІН...,Accept,0.644278,SDGs and Well-being



✓ REVERSE STRATEGY (ONE-VS-ALL) COMPLETE!


In [11]:
display(test_df[test_df['Prediction']=='Reject'][output_cols])

,ID_New,Article Title,Prediction,Confidence,Theme
6,OA_1940,"""The farm has an insatiable appetite"": A food ...",Reject,0.491827,
8,Scopus_1613,"""When you have stress because you don't have f...",Reject,0.295621,
9,OA_3763,#36915 D37 – the green footprint of regional a...,Reject,0.420379,
10,OA_5660,(210) Breastfeeding: The Blind Spot in Sexuali...,Reject,0.440962,
14,WoS_1491,"?When I say I?m depressed, it?s like anger.? A...",Reject,0.353501,
...,...,...,...,...,...
10167,WoS_6128,Zinc zeolite nanoparticle-modified adhesive re...,Reject,0.555704,
10168,OA_3189,Zoonotic infectious diseases as ecosystem diss...,Reject,0.601597,
10172,OA_4940,Պարենային անվտանգության ոլորտում ԵԱՏՄ երկրների...,Reject,0.459168,
10173,OA_4085,آثار تغير المناخ على الاقتصاد الأزرق في مصر,Reject,0.368791,


## Analysis: Compare with Training Distribution

In [12]:
print("\n" + "="*80)
print("COMPARISON: TRAINING vs TEST DISTRIBUTION")
print("="*80)

# Training distribution
train_theme_counts = accepted_df['If Accept, identify theme'].value_counts()
train_total = len(accepted_df)

# Test distribution
test_accept = test_df[test_df['Prediction']=='Accept']
test_theme_counts = test_accept['Theme'].value_counts()
test_total = len(test_accept)

print(f"\n{'Theme':<60} | Training | Test")
print(f"{'-'*60}|----------|------")

for theme in theme_encoder.classes_:
    train_pct = (train_theme_counts.get(theme, 0) / train_total * 100) if train_total > 0 else 0
    test_pct = (test_theme_counts.get(theme, 0) / test_total * 100) if test_total > 0 else 0
    
    print(f"{theme:<60} | {train_pct:6.1f}% | {test_pct:6.1f}%")

print(f"\n{'='*80}")
print("\n✓ If test distribution roughly matches training, the model is working well!")
print("✓ Small differences are expected due to different paper content.")
print("="*80)


COMPARISON: TRAINING vs TEST DISTRIBUTION

Theme                                                        | Training | Test
------------------------------------------------------------|----------|------
Access to Services and Wellbeing                             |    3.0% |    0.8%
Better Well-being Metric                                     |    3.5% |    1.4%
Climate change mitigation options & Well-being               |   42.4% |    5.1%
Ecosystem & Non-human Well-being                             |    3.0% |    0.3%
Empowering People & Governance, Policy                       |   15.7% |    2.5%
Equity, Social Justice, Just Transition                      |    4.0% |   36.6%
Implemented mitigation Options and Well-being                |   10.6% |    0.2%
Integrated Responses and Well-being                          |    3.0% |    0.7%
SDGs and Well-being                                          |    5.1% |   14.9%
Systems Thinking- Modelling and Supply Chain                 |    1.5

In [13]:
df=pd.read_csv("/kaggle/working/reverse_strategy_predictions.csv")
df

,ID_New,Article Title,Prediction,Confidence,Theme
0,OA_3712,NaN,Accept,0.857219,Systems Thinking- Modelling and Supply Chain
1,WoS_1385,"It ' s one thing after another, after another...",Accept,0.917544,"Equity, Social Justice, Just Transition"
2,Scopus_5109,"""A Return to and of the Land"": Indigenous Know...",Accept,0.812806,"Equity, Social Justice, Just Transition"
3,Scopus_4859,"""I see my culture starting to disappear"": Anis...",Accept,0.989308,"Equity, Social Justice, Just Transition"
4,Scopus_1176,"""Impact of Climate Change on Coastal Cities: A...",Accept,0.971161,"Equity, Social Justice, Just Transition"
...,...,...,...,...,...
10170,OA_4235,ҚАЗАҚСТАН РЕСПУБЛИКАСЫНДАҒЫ АЗЫҚ-ТҮЛІК ЖҮЙЕСІН...,Accept,0.644278,SDGs and Well-being
10171,OA_3078,СУЧАСНИЙ СТАН БАНКІВСЬКОЇ СИСТЕМИ РЕСПУБЛІКИ К...,Accept,0.710149,SDGs and Well-being
10172,OA_4940,Պարենային անվտանգության ոլորտում ԵԱՏՄ երկրների...,Reject,0.459168,NaN
10173,OA_4085,آثار تغير المناخ على الاقتصاد الأزرق في مصر,Reject,0.368791,NaN


In [15]:
# Add missing imports
from sklearn.metrics import precision_score, recall_score, confusion_matrix, classification_report

print("\n" + "="*80)
print("BINARY CLASSIFICATION METRICS (ACCEPT vs REJECT)")
print("="*80)

# ============================================
# PART 1: OOF Predictions on Training Data
# ============================================

print("\n1. TRAINING DATA EVALUATION (Out-of-Fold)")
print("-"*80)

# We need to get OOF predictions for the training data
# Run all 12 classifiers on training data to get aggregated predictions

train_dataset_full = ClimateDataset(
    train_df['text'].values,
    np.zeros(len(train_df)),  # Dummy labels
    tokenizer,
    CFG.max_length
)

train_loader_full = DataLoader(
    train_dataset_full,
    batch_size=CFG.batch_size*2,
    shuffle=False,
    num_workers=CFG.num_workers
)

# Get predictions from all 12 classifiers on full training data
train_theme_probs = np.zeros((len(train_df), num_themes))

print("\nGenerating OOF predictions on training data...")

for theme_id, clf_info in enumerate(all_theme_classifiers):
    print(f"  Theme {theme_id}...", end='')
    
    theme_probs_all_folds = []
    
    for model in clf_info['models']:
        model.to(CFG.device)
        model.eval()
        
        theme_probs = []
        
        with torch.no_grad():
            for batch in train_loader_full:
                input_ids = batch['input_ids'].to(CFG.device)
                attention_mask = batch['attention_mask'].to(CFG.device)
                
                logits = model(input_ids, attention_mask)
                probs = F.softmax(logits, dim=1).cpu().numpy()
                theme_probs.append(probs[:, 1])
        
        theme_probs_all_folds.append(np.concatenate(theme_probs))
        
        model.cpu()
        torch.cuda.empty_cache()
    
    train_theme_probs[:, theme_id] = np.mean(theme_probs_all_folds, axis=0)
    print(" Done")

# Apply same decision logic as test
train_max_probs = np.max(train_theme_probs, axis=1)
train_max_theme_ids = np.argmax(train_theme_probs, axis=1)

train_predictions = []
for i in range(len(train_df)):
    max_prob = train_max_probs[i]
    max_theme_id = train_max_theme_ids[i]
    theme_threshold = all_theme_classifiers[max_theme_id]['avg_threshold']
    
    if max_prob > theme_threshold and max_prob > CFG.accept_threshold:
        train_predictions.append(1)  # Accept
    else:
        train_predictions.append(0)  # Reject

train_predictions = np.array(train_predictions)

# Ground truth
train_true_labels = (train_df['Accept/Reject'] == 'Accept').astype(int).values

# Calculate metrics
train_f1 = f1_score(train_true_labels, train_predictions, average='binary')
train_acc = accuracy_score(train_true_labels, train_predictions)
train_precision = precision_score(train_true_labels, train_predictions, average='binary', zero_division=0)
train_recall = recall_score(train_true_labels, train_predictions, average='binary', zero_division=0)

print("\n" + "="*80)
print("TRAINING DATA RESULTS (OOF)")
print("="*80)

print(f"\nBinary Classification Metrics:")
print(f"  Accuracy:  {train_acc:.4f}")
print(f"  Precision: {train_precision:.4f}")
print(f"  Recall:    {train_recall:.4f}")
print(f"  F1 Score:  {train_f1:.4f}")

print(f"\nConfusion Matrix:")
cm = confusion_matrix(train_true_labels, train_predictions)
print(f"                Predicted")
print(f"              Reject  Accept")
print(f"True Reject:  {cm[0,0]:5d}  {cm[0,1]:5d}")
print(f"True Accept:  {cm[1,0]:5d}  {cm[1,1]:5d}")

print(f"\nDetailed Classification Report:")
print(classification_report(
    train_true_labels,
    train_predictions,
    target_names=['Reject', 'Accept'],
    digits=4,
    zero_division=0
))

# ============================================
# PART 2: Test Data Statistics
# ============================================

print("\n" + "="*80)
print("2. TEST DATA PREDICTIONS (No Ground Truth)")
print("="*80)

# Load the predictions file
test_predictions_df = pd.read_csv(f'{CFG.output_dir}/reverse_strategy_predictions.csv')

n_test_accept = (test_predictions_df['Prediction'] == 'Accept').sum()
n_test_reject = (test_predictions_df['Prediction'] == 'Reject').sum()
n_test_total = len(test_predictions_df)

print(f"\nTest Set Predictions:")
print(f"  Total papers:     {n_test_total:,}")
print(f"  Predicted Accept: {n_test_accept:,} ({n_test_accept/n_test_total*100:.2f}%)")
print(f"  Predicted Reject: {n_test_reject:,} ({n_test_reject/n_test_total*100:.2f}%)")

# Confidence statistics
accept_rows = test_predictions_df[test_predictions_df['Prediction'] == 'Accept']
reject_rows = test_predictions_df[test_predictions_df['Prediction'] == 'Reject']

if len(accept_rows) > 0:
    print(f"\nAccept Predictions - Confidence Stats:")
    print(f"  Mean:   {accept_rows['Confidence'].mean():.4f}")
    print(f"  Median: {accept_rows['Confidence'].median():.4f}")
    print(f"  Min:    {accept_rows['Confidence'].min():.4f}")
    print(f"  Max:    {accept_rows['Confidence'].max():.4f}")

if len(reject_rows) > 0:
    print(f"\nReject Predictions - Confidence Stats:")
    print(f"  Mean:   {reject_rows['Confidence'].mean():.4f}")
    print(f"  Median: {reject_rows['Confidence'].median():.4f}")
    print(f"  Min:    {reject_rows['Confidence'].min():.4f}")
    print(f"  Max:    {reject_rows['Confidence'].max():.4f}")

# ============================================
# PART 3: Compare Training vs Test Distributions
# ============================================

print("\n" + "="*80)
print("3. DISTRIBUTION COMPARISON")
print("="*80)

# Training distribution
train_accept_rate = (train_true_labels == 1).sum() / len(train_true_labels) * 100
test_accept_rate = n_test_accept / n_test_total * 100

print(f"\nAccept Rate Comparison:")
print(f"  Training (True):      {train_accept_rate:.2f}%")
print(f"  Training (Predicted): {(train_predictions == 1).sum() / len(train_predictions) * 100:.2f}%")
print(f"  Test (Predicted):     {test_accept_rate:.2f}%")

if abs(train_accept_rate - test_accept_rate) < 5:
    print(f"\n✓ Good! Test accept rate is close to training distribution.")
elif test_accept_rate < train_accept_rate - 5:
    print(f"\n⚠️ Warning: Test accept rate is lower than training. Model might be too conservative.")
    print(f"   Consider lowering accept_threshold from {CFG.accept_threshold} to ~{CFG.accept_threshold - 0.05:.2f}")
elif test_accept_rate > train_accept_rate + 5:
    print(f"\n⚠️ Warning: Test accept rate is higher than training. Model might be too permissive.")
    print(f"   Consider increasing accept_threshold from {CFG.accept_threshold} to ~{CFG.accept_threshold + 0.05:.2f}")

print("\n" + "="*80)


BINARY CLASSIFICATION METRICS (ACCEPT vs REJECT)

1. TRAINING DATA EVALUATION (Out-of-Fold)
--------------------------------------------------------------------------------

Generating OOF predictions on training data...
  Theme 0... Done
  Theme 1... Done
  Theme 2... Done
  Theme 3... Done
  Theme 4... Done
  Theme 5... Done
  Theme 6... Done
  Theme 7... Done
  Theme 8... Done
  Theme 9... Done
  Theme 10... Done
  Theme 11... Done

TRAINING DATA RESULTS (OOF)

Binary Classification Metrics:
  Accuracy:  0.4061
  Precision: 0.1493
  Recall:    0.8794
  F1 Score:  0.2553

Confusion Matrix:
                Predicted
              Reject  Accept
True Reject:    523    997
True Accept:     24    175

Detailed Classification Report:
              precision    recall  f1-score   support

      Reject     0.9561    0.3441    0.5060      1520
      Accept     0.1493    0.8794    0.2553       199

    accuracy                         0.4061      1719
   macro avg     0.5527    0.6117    0.3